![image.png](https://i.imgur.com/a3uAqnb.png)

# Responsible AI Lab: Audit, Explain and Document a Model

> **Optional lab.** The worked solution is `04_Responsible_AI_Audit_Solution.ipynb`, in this folder.

This lab is the practical half of **Lecture 01 - Responsible AI and Model Interpretability**. The deck proposes three exercises: *"disaggregate on UCI Adult, show the parity/odds tension, apply threshold post-processing, and measure what it cost"*, *"run SHAP and LIME on the same tabular model"*, and *"write a model card"*. Here they are in the order a real audit runs:

**frame the harm → measure per group → find the conflict → test the proxies → mitigate → explain → document → decide**

**The scenario** is invented for teaching, like the deck's credit-model card. A bank wants a model to **pre-screen applicants for a credit product**. The only labelled data it has is the 1994 US Census, where the label is *"earned more than $50K a year"*. You are the reviewer who has to sign the model off, or refuse to.

### What you will have by the end

- A **disaggregated report** with sample sizes and error bars: the deck's "single most valuable habit".
- Your own numbers showing why the **fairness definitions conflict**, and why **dropping the sensitive attribute does not work**.
- A **post-processing fix**, and what it cost.
- **Explanations** you can defend, and two you should not trust.
- A **model card** generated from the code, and a **governance decision** written into it.

**Time:** 75-90 minutes, CPU only. No accounts are needed.

## Setup

This lab adds three libraries: **fairlearn** for fairness metrics and mitigation, and **shap** and **lime** for explanations. Colab already ships numpy, pandas, scikit-learn and matplotlib.

In [ ]:
# The libraries this lab adds, pinned to the versions it was tested with (October 2026).
%pip install -q "fairlearn==0.14.0" "shap==0.52.0" "lime==0.2.0.1"

In [ ]:
import warnings
from datetime import date, timedelta
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)   # library deprecation notices, not lab output

SEED = 42   # every random_state below uses it
for lib in ["numpy", "pandas", "scikit-learn", "fairlearn", "shap", "lime"]:
    print(f"{lib:<13} {version(lib)}")

# Content

## 1. The data, and what its label measures

> **From the deck, *Measurement Bias: The Label Is a Proxy*:** "*My label is X. What exactly had to happen in the world for X to be recorded, and does that differ by group?*"

**UCI Adult** ([Becker & Kohavi, 1996](https://doi.org/10.24432/C5XW20)) has 48,842 people from the 1994 US Census, with age, education, occupation, hours worked and more. The label is whether the person's income was above $50K.

As a credit-screening label it is a proxy twice over. Income is not repayment. And incomes recorded in 1994 carry the labour market of 1994: the deck's **historical bias**, "*present even with perfect sampling and perfect measurement*". We use it because it is public, well studied, and has the attributes an audit needs.

The data ships with the course in `data/adult.csv.gz`. If the file is not there (for example on Colab), the cell downloads the same table from OpenML.

In [ ]:
candidates = [Path("data/adult.csv.gz"), Path("../data/adult.csv.gz")]
local = next((p for p in candidates if p.exists()), None)
if local is not None:
    raw = pd.read_csv(local)
else:
    from sklearn.datasets import fetch_openml
    raw = fetch_openml(data_id=1590, as_frame=True, parser="auto").frame   # the same table, from OpenML
    raw = raw.astype({c: object for c in raw.select_dtypes("category").columns})

# fnlwgt is a census sampling weight, not a fact about the person; education duplicates education-num.
df = raw.drop(columns=["fnlwgt", "education"])
y = (df.pop("class") == ">50K").astype(int)

CATEGORICAL = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
for c in CATEGORICAL:
    df[c] = df[c].fillna("Unknown").astype("category")   # a missing answer is information too: keep it as a category

print(f"{len(df):,} people, {df.shape[1]} features; {y.mean():.1%} earn more than $50K")
df.head()

> **From the deck, *Historical and Representation Bias*:** "*More targeted data does help, but you must first know the composition of your data.*"

Before training anything, look at who is in the data and how often each group has the positive label.

In [ ]:
def composition(column):
    groups = df.groupby(column, observed=True)
    return pd.DataFrame({"n": groups.size(),
                         "share of data": groups.size() / len(df),
                         "base rate (>$50K)": y.groupby(df[column], observed=True).mean()}).round(3)

display(composition("sex"))
display(composition("race"))

Two facts drive the rest of the lab:

- **Representation.** There is one woman for every two men, and some race groups are tiny: a few hundred people each. Any metric computed on those groups will be noisy.
- **Base rates differ.** About 30% of men are labelled positive, against 11% of women. The deck's impossibility result needs exactly this condition: *when base rates differ, calibration and error-rate balance cannot both hold*. You will see it in your own numbers in section 4.

## 2. A model, and its headline number

> **From the deck, *Try an Interpretable Model First* (Rudin, 2019):** "*On many tabular problems, a well-regularised logistic regression, a depth-4 tree, or a generalised additive model is within a point or two of a gradient-boosted ensemble. Measure the gap before assuming you need the black box.*"

We split the data three ways:

- **train**, to fit the models;
- **validation**, to choose decision thresholds in section 6;
- **test**, touched only to audit.

The models get categorical columns as **integer codes**, and the gradient-boosted model is told which columns are categorical. LIME and SHAP need a numeric table too. Whenever a person reads the output, we map the codes back to their labels.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.copy()
for c in CATEGORICAL:
    X[c] = df[c].cat.codes   # labels stay in df[c].cat.categories

idx = np.arange(len(X))
i_train, i_rest = train_test_split(idx, test_size=0.4, stratify=y, random_state=SEED)
i_val, i_test = train_test_split(i_rest, test_size=0.5, stratify=y.iloc[i_rest], random_state=SEED)
X_train, X_val, X_test = X.iloc[i_train], X.iloc[i_val], X.iloc[i_test]
y_train, y_val, y_test = y.iloc[i_train], y.iloc[i_val], y.iloc[i_test]

A_val, A_test = df["sex"].iloc[i_val], df["sex"].iloc[i_test]   # the sensitive attribute, A in the deck's notation
print(f"train {len(i_train):,} | validation {len(i_val):,} | test {len(i_test):,}")

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

IS_CATEGORICAL = [c in CATEGORICAL for c in X.columns]
NUMERIC = [c for c in X.columns if c not in CATEGORICAL]

model = HistGradientBoostingClassifier(categorical_features=IS_CATEGORICAL, random_state=SEED).fit(X_train, y_train)
glass_box = make_pipeline(
    make_column_transformer((OneHotEncoder(handle_unknown="ignore"), CATEGORICAL), (StandardScaler(), NUMERIC)),
    LogisticRegression(max_iter=2000),
).fit(X_train, y_train)

for name, m in [("gradient-boosted trees", model), ("logistic regression", glass_box)]:
    p = m.predict_proba(X_test)[:, 1]
    print(f"{name:<23} accuracy {accuracy_score(y_test, p >= 0.5):.3f}   AUC {roc_auc_score(y_test, p):.3f}")

p_test = model.predict_proba(X_test)[:, 1]
y_pred = (p_test >= 0.5).astype(int)   # the decision: pre-approve if the score is at least 0.5

The gap is about two AUC points. We continue with the boosted model, because that is what most teams would ship, and because it gives the explanation tools something to do.

Whether two points justify a black box for a credit decision is a judgement, and it belongs in the model card (section 8), not in someone's memory.

## 3. Disaggregated evaluation

> **From the deck, *Disaggregated Evaluation: The Core Technique*:** "*One number for the test set, then the same number for every slice you can define. ... Always print n beside the metric. Always print the worst slice in the summary, not just the mean.*"

fairlearn's `MetricFrame` is the deck's "*groupby plus classification report*". We report the rates the deck defines on its *Object Everything Is Computed From* slide:

- the **selection rate**, $\Pr(\hat{Y}=1)$;
- the **TPR**, $\Pr(\hat{Y}=1 \mid Y=1)$;
- the **FPR**, $\Pr(\hat{Y}=1 \mid Y=0)$;
- the **PPV**, $\Pr(Y=1 \mid \hat{Y}=1)$.

In [ ]:
from fairlearn.metrics import MetricFrame, false_positive_rate, selection_rate, true_positive_rate
from sklearn.metrics import precision_score

METRICS = {
    "n": lambda yt, yp: len(yt),
    "positives": lambda yt, yp: int(np.sum(yt)),
    "selection rate": selection_rate,
    "TPR": true_positive_rate,
    "FPR": false_positive_rate,
    "PPV": lambda yt, yp: precision_score(yt, yp, zero_division=0),
}
by_sex = MetricFrame(metrics=METRICS, y_true=y_test, y_pred=y_pred, sensitive_features=A_test)
print("Overall:")
display(by_sex.overall.to_frame().T.round(3))
print("By sex:")
display(by_sex.by_group.round(3))

Now the intersections. Gender Shades found the largest gaps there, and the deck warns that "*the overall row is fine ... the intersectional rows are worse*".

> **From the deck, *Four Things That Go Wrong in Practice*:** "*Small slices are noisy. A TPR of 0.44 estimated from 180 positive cases carries a 95% interval of about ±0.07; from 40 cases, about ±0.15. Report intervals, or you will chase noise. A bootstrap is fine here.*"

So every slice gets a 95% **bootstrap interval** for its TPR: resample the slice's rows with replacement many times, recompute the TPR each time, and report the middle 95% of the results.

In [ ]:
def tpr(yt, yp):
    return yp[yt == 1].mean()

def bootstrap_interval(yt, yp, metric=tpr, n_boot=1000, seed=SEED):
    # 95% percentile interval of `metric` over resamples of one slice.
    yt, yp = np.asarray(yt), np.asarray(yp)
    rng = np.random.default_rng(seed)
    stats = []
    for _ in range(n_boot):
        i = rng.integers(0, len(yt), len(yt))
        if yt[i].sum() > 0:   # the TPR is undefined in a resample with no positives
            stats.append(metric(yt[i], yp[i]))
    return np.percentile(stats, [2.5, 97.5])

slices = df[["sex", "race"]].iloc[i_test].astype(str).agg(" / ".join, axis=1).values
rows = []
for s in sorted(set(slices)):
    m = slices == s
    yt, yp = y_test.values[m], y_pred[m]
    low, high = bootstrap_interval(yt, yp)
    rows.append({"slice": s, "n": int(m.sum()), "positives": int(yt.sum()), "TPR": tpr(yt, yp), "95% low": low, "95% high": high})
tpr_by_slice = pd.DataFrame(rows).set_index("slice").round(3)
tpr_by_slice

Read the interval column before the TPR column. A slice with a handful of positives can show any TPR, from 0 to 1, by chance, and its interval says so.

> **Exercise (TASK 1):** the deck says to "*print the worst slice in the summary*", but the worst slice by raw TPR is usually a tiny one. Find the **worst slice you can trust**: the lowest TPR among slices with at least `min_positives` positive cases. Return its row from `tpr_by_slice`. If you skip it, the rest of the lab still runs.

In [ ]:
def worst_reliable_slice(table, min_positives=25):
    # TASK 1: keep the rows of `table` with at least `min_positives` positives, then return the
    # row (a pandas Series) with the lowest TPR.
    # HINT: filter with table[table["positives"] >= min_positives]; the row is .loc[filtered["TPR"].idxmin()].
    raise NotImplementedError("TASK 1: implement worst_reliable_slice")

try:
    worst = worst_reliable_slice(tpr_by_slice)
    print(f"Worst reliable slice: {worst.name}, TPR {worst['TPR']:.3f} "
          f"(95% interval {worst['95% low']:.2f}-{worst['95% high']:.2f}, {int(worst['positives'])} positives); "
          f"overall TPR {by_sex.overall['TPR']:.3f}")
except NotImplementedError as e:
    worst = None
    print(f"{e}: skipped. The rest of the lab does not depend on it.")

## 4. The conflict, in your own numbers

> **From the deck, *You Cannot Have All Three* (Chouldechova, 2017, eq. 2.6):** $\text{FPR} = \dfrac{p}{1-p} \cdot \dfrac{1-\text{PPV}}{\text{PPV}} \cdot (1-\text{FNR})$. "*Fix PPV to be equal across two groups. If $p_A \neq p_B$, the right-hand side differs, so FPR and FNR cannot both be equal. ... It is arithmetic.*"

First, check that the identity holds exactly for each group's confusion matrix:

In [ ]:
rows = []
for g in ["Female", "Male"]:
    m = (A_test == g).values
    yt, yp = y_test.values[m], y_pred[m]
    tp, fp = np.sum((yt == 1) & (yp == 1)), np.sum((yt == 0) & (yp == 1))
    fn, tn = np.sum((yt == 1) & (yp == 0)), np.sum((yt == 0) & (yp == 0))
    p, ppv, fnr, fpr = yt.mean(), tp / (tp + fp), fn / (fn + tp), fp / (fp + tn)
    rows.append({"group": g, "prevalence p": p, "PPV": ppv, "FNR": fnr,
                 "FPR (measured)": fpr, "FPR (from the identity)": p / (1 - p) * (1 - ppv) / ppv * (1 - fnr)})
pd.DataFrame(rows).set_index("group").round(4)

The identity holds to the last digit, so with prevalences of about 0.30 and 0.12, no single threshold can equalise everything. At the 0.5 threshold:

- **independence** (demographic parity) compares selection rates;
- **separation** (equalised odds) compares TPR and FPR;
- **sufficiency** (predictive parity) compares PPV.

In [ ]:
from fairlearn.metrics import demographic_parity_ratio

g = by_sex.by_group
print(f"Demographic parity : selection-rate ratio {demographic_parity_ratio(y_test, y_pred, sensitive_features=A_test):.2f}"
      "  (the deck's 0.8 convention flags anything below 0.8)")
print(f"Equalised odds     : TPR gap {abs(g.loc['Female', 'TPR'] - g.loc['Male', 'TPR']):.3f}, "
      f"FPR gap {abs(g.loc['Female', 'FPR'] - g.loc['Male', 'FPR']):.3f}")
print(f"Predictive parity  : PPV {g.loc['Female', 'PPV']:.3f} (women) vs {g.loc['Male', 'PPV']:.3f} (men)")

> **From the deck, *Calibration and Predictive Parity*:** "*Check it with a reliability diagram per group, not with a single number.*"

A reliability diagram puts the mean score in each bin on the x-axis and the share of that bin that really earned more than $50K on the y-axis. On the diagonal, a score of 0.7 means a 70% chance for everyone.

In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for group, color in [("Female", "#d03b3b"), ("Male", "#2a78d6")]:
    m = (A_test == group).values
    observed, predicted = calibration_curve(y_test[m], p_test[m], n_bins=8, strategy="uniform")
    ax.plot(predicted, observed, "o-", color=color, label=f"{group} (n = {m.sum():,})")
ax.plot([0, 1], [0, 1], ":", color="gray", label="calibrated")
ax.set_xlabel("mean score in bin")
ax.set_ylabel("share that earned > $50K")
ax.set_title("Reliability by group", loc="left")
ax.legend(frameon=False)
plt.show()

Both curves follow the diagonal at the extremes, where most people are. Look at the middle, where the threshold sits. If the women's curve runs **above** the men's there, a woman and a man with the same mid-range score do not have the same chance of the outcome: the score means different things for the two groups in exactly the range where the decision is made.

So on this data the model fails all three criteria at once. The next two sections ask what can be done about it.

## 5. "Just drop the sensitive attribute"

> **From the deck, *"Just Drop the Sensitive Attribute" Does Not Work*:** "*A capable model reconstructs the attribute from the proxies. Removing the column removes your ability to measure the disparity, not the disparity itself.*"

The first test: retrain without `sex` and compare the selection rates.

In [ ]:
blind = HistGradientBoostingClassifier(categorical_features=[c in CATEGORICAL for c in X.columns if c != "sex"],
                                       random_state=SEED).fit(X_train.drop(columns="sex"), y_train)
y_pred_blind = blind.predict(X_test.drop(columns="sex"))

for name, yp in [("with sex", y_pred), ("without sex", y_pred_blind)]:
    rates = pd.Series(yp).groupby(A_test.values, observed=True).mean()
    print(f"{name:<12} selection rate: women {rates['Female']:.3f}, men {rates['Male']:.3f}, "
          f"ratio {rates['Female'] / rates['Male']:.2f}; accuracy {accuracy_score(y_test, yp):.3f}")

Almost nothing moved. To see why, test the deck's mechanism directly: if the other features can predict `sex`, they are proxies for it.

> **Exercise (TASK 2):** train a classifier to predict `sex` from **every other feature**, and report its test AUC. Use `HistGradientBoostingClassifier` with the categorical mask for the remaining columns. An AUC near 0.5 would mean no proxies; near 1.0 means the model can rebuild `sex` almost perfectly.

In [ ]:
def proxy_auc(drop=("sex",)):
    # TASK 2: predict "is female" from every column of X except those in `drop`; return the test AUC.
    # HINT: target_train = (df["sex"].iloc[i_train] == "Female").astype(int), and the same for i_test.
    #       Fit HistGradientBoostingClassifier(categorical_features=[c in CATEGORICAL for c in cols], random_state=SEED)
    #       on X_train[cols], then roc_auc_score on X_test[cols].
    raise NotImplementedError("TASK 2: implement proxy_auc")

try:
    print(f"AUC predicting sex from the other features             : {proxy_auc():.3f}")
    print(f"... and with 'relationship' (Husband/Wife) also removed: {proxy_auc(drop=('sex', 'relationship')):.3f}")
except NotImplementedError as e:
    print(f"{e}: skipped. The rest of the lab does not depend on it.")

`relationship` takes the values *Husband* and *Wife*, so it is `sex` by another name. Even without it, marital status, occupation and hours worked rebuild most of the signal.

This is the deck's corollary: "*you often need the sensitive attribute, held securely and used only for auditing, in order to check for harm*". Without the `sex` column, section 3 could not have been written.

## 6. Mitigation: group-specific thresholds

> **From the deck, *Post-processing: Group-Specific Thresholds*:** "*Hardt et al. (2016) show that if you keep the score R and choose a separate threshold per group, you can achieve equalised odds.*" And: "*Treat it as a decision to be agreed and recorded, not as a parameter to be tuned.*"

fairlearn's `ThresholdOptimizer` keeps our trained model and picks thresholds per group so that one constraint holds. We try three of them:

- **demographic parity:** equal selection rates;
- **true-positive-rate parity:** *equal opportunity*, so people who would succeed have the same chance in each group;
- **equalised odds:** equal TPR *and* equal FPR.

The thresholds are fitted on the **validation** split and the result is measured on **test**: "*guard the test set*".

In [ ]:
from fairlearn.postprocessing import ThresholdOptimizer, plot_threshold_optimizer

def rates(yp):
    mf = MetricFrame(metrics={"sel": selection_rate, "TPR": true_positive_rate, "FPR": false_positive_rate},
                     y_true=y_test, y_pred=yp, sensitive_features=A_test).by_group
    return {"accuracy": accuracy_score(y_test, yp),
            "selection F/M": f"{mf.loc['Female', 'sel']:.3f} / {mf.loc['Male', 'sel']:.3f}",
            "TPR F/M": f"{mf.loc['Female', 'TPR']:.3f} / {mf.loc['Male', 'TPR']:.3f}",
            "FPR F/M": f"{mf.loc['Female', 'FPR']:.3f} / {mf.loc['Male', 'FPR']:.3f}"}

results = {"no constraint (0.5 for everyone)": rates(y_pred)}
optimizers = {}
for constraint in ["demographic_parity", "true_positive_rate_parity", "equalized_odds"]:
    opt = ThresholdOptimizer(estimator=model, constraints=constraint, objective="accuracy_score",
                             prefit=True, predict_method="predict_proba")
    opt.fit(X_val, y_val, sensitive_features=A_val)
    optimizers[constraint] = opt
    results[constraint] = rates(opt.predict(X_test, sensitive_features=A_test, random_state=SEED))

pd.DataFrame(results).T.round(3)

Read it row by row:

- **Demographic parity** equalises selection rates, at the largest accuracy cost. Look at the TPRs: the error-rate gap does not close, it **reverses**. That is section 4's arithmetic showing up as a decision.
- **Equal opportunity** brings the TPRs together, here at almost no accuracy cost; the FPRs stay apart.
- **Equalised odds** brings both together, and gives up a little accuracy.

The plot shows each group's ROC curve and the operating point equalised odds chose: the deck's *reachable by both* construction.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
plot_threshold_optimizer(optimizers["equalized_odds"], ax=ax, show_plot=False)
ax.set_title("Equalised odds: one operating point both groups can reach", loc="left", fontsize=10)
plt.show()

> **Before you reach for this.** Two applicants with the same score can now get different decisions because of their sex, and `ThresholdOptimizer` may even **randomise** between two thresholds to hit a target exactly. The deck says to "*say so out loud*". In many settings that alone rules it out, and the fix moves to the data or the label.

For the rest of the lab we adopt **equal opportunity**, and section 8 writes that choice, and what it gives up, into the model card. It is a judgement, not a result. If you would choose differently, change `CHOSEN` and argue for it.

In [ ]:
CHOSEN = "true_positive_rate_parity"
y_fair = optimizers[CHOSEN].predict(X_test, sensitive_features=A_test, random_state=SEED)
print(f"Adopted: {CHOSEN}; test accuracy {accuracy_score(y_test, y_fair):.3f} (was {accuracy_score(y_test, y_pred):.3f})")

## 7. Explanations you can defend, and two you should not trust

### 7a. A trap: built-in tree importance

> **From the deck, *A Trap: Built-in Tree "Feature Importance"*:** "*Add a column of random integers to your dataset and it can outrank real predictors in MDI, while permutation importance on held-out data correctly gives it roughly zero.*"

Here is that demonstration. A random forest gets one extra column, `random_id`, that carries no information at all.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

noise = np.random.default_rng(SEED)
X_train_id = X_train.assign(random_id=noise.integers(0, 100_000, len(X_train)))
X_test_id = X_test.assign(random_id=noise.integers(0, 100_000, len(X_test)))

forest = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED).fit(X_train_id, y_train)
mdi = pd.Series(forest.feature_importances_, index=X_train_id.columns)
perm = permutation_importance(forest, X_test_id.iloc[:3000], y_test.iloc[:3000], n_repeats=5,
                              random_state=SEED, n_jobs=-1).importances_mean
both = pd.DataFrame({"MDI (training data)": mdi, "permutation (held-out)": pd.Series(perm, index=X_train_id.columns)})
both["MDI rank"] = both.iloc[:, 0].rank(ascending=False).astype(int)
both["permutation rank"] = both.iloc[:, 1].rank(ascending=False).astype(int)
both.sort_values("MDI rank").round(4)

The column of noise is near the top by MDI and near the bottom by permutation importance.

MDI is computed on the **training** data and rewards **high-cardinality** columns: a random ID offers thousands of split points, and the forest uses them to memorise. The deck's rule: never report "*the model's feature importance*" without saying which definition you used.

### 7b. Permutation importance for our model

> **From the deck, *Baseline: Permutation Importance*:** "*Shuffle one feature column, breaking its relationship with the target. Re-score. The drop is that feature's importance.*"

In [ ]:
perm_model = permutation_importance(model, X_test.iloc[:3000], y_test.iloc[:3000], scoring="roc_auc",
                                    n_repeats=5, random_state=SEED, n_jobs=-1)
(pd.Series(perm_model.importances_mean, index=X.columns).sort_values()
   .plot.barh(figsize=(6, 4), color="#2a78d6", title="Drop in test AUC when the column is shuffled"))
plt.show()

### 7c. SHAP: one applicant, then everyone

> **From the deck, *SHAP: Attribution as a Fair Payout*:** each feature's Shapley value is its average contribution, and the values add up: "*base value + Σ φ = f(x)*".

The deck's *Which SHAP Estimator?* slide recommends TreeSHAP for tree models. This model splits on **categories**, though, and SHAP's tree explainer does not read those splits correctly. So we use the model-agnostic **permutation** explainer instead, and check the additivity property ourselves rather than trusting it. It is slower, so we explain 150 applicants (about half a minute).

In [ ]:
import shap

background = X_train.sample(100, random_state=SEED)   # defines E[f(X)], "the typical applicant"
explainer = shap.Explainer(lambda Z: model.predict_proba(pd.DataFrame(Z, columns=X.columns))[:, 1],
                           shap.maskers.Independent(background, max_samples=100),
                           algorithm="permutation", seed=SEED)
sample = X_test.iloc[:150]
sv = explainer(sample, silent=True)
sv.display_data = df.iloc[i_test[:150]].astype(object).values   # show labels, not integer codes

adds_up = np.allclose(sv.base_values + sv.values.sum(axis=1), model.predict_proba(sample)[:, 1], atol=1e-6)
print(f"base value + sum of SHAP values == model output for all 150 applicants: {adds_up}")

In [ ]:
applicant = 0
print(f"Applicant {applicant}: score {p_test[applicant]:.3f}, decision {'pre-approve' if y_pred[applicant] else 'decline'}, "
      f"true label {'>$50K' if y_test.iloc[applicant] else '<=$50K'}")
shap.plots.waterfall(sv[applicant], max_display=10)

In [ ]:
shap.plots.beeswarm(sv, max_display=12)

Read the waterfall the way the deck suggests putting it to a stakeholder: "*compared with a typical applicant, this applicant's ... pushed the score up by ...*".

Then look for `sex` in both plots. The model uses it directly, and `relationship` carries more of the same signal.

### 7d. LIME, five times

> **From the deck, *Pitfalls You Must Know Before Using Either*:** "*Instability. Re-running LIME with a different seed can reorder the top features. Always run it several times and check the explanation is stable before you show it to anyone.*"

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

cat_idx = [i for i, c in enumerate(X.columns) if c in CATEGORICAL]
predict = lambda Z: model.predict_proba(pd.DataFrame(Z, columns=X.columns))
row = X_test.iloc[applicant].values.astype(float)

tops = {}
for seed in range(5):
    lime_explainer = LimeTabularExplainer(
        X_train.values.astype(float), feature_names=list(X.columns), class_names=["<=50K", ">50K"],
        categorical_features=cat_idx, categorical_names={i: list(df[X.columns[i]].cat.categories) for i in cat_idx},
        discretize_continuous=True, random_state=seed)
    explanation = lime_explainer.explain_instance(row, predict, num_features=5, num_samples=2000)
    tops[f"seed {seed}"] = [feature for feature, weight in explanation.as_list()]

display(pd.DataFrame(tops, index=[f"#{k}" for k in range(1, 6)]))
distinct = {f for features in tops.values() for f in features}
print(f"{len(distinct)} different features appear in the top 5 across 5 runs on the same applicant.")

If the five columns are not identical, you have seen the instability for yourself. LIME's weights also depend on how it **discretised** the continuous features (`hours-per-week <= 40.00`), which is a choice it made, not a fact about the model.

> **From the deck, *Explaining the Model vs Explaining the World*:** SHAP answers "*how does this fitted function distribute its output across the inputs I gave it?*". People hear "*which factors cause this outcome, and what should this person change?*". The second is a causal question.

If this applicant asked what to change to be approved, neither plot answers it. That needs a **counterfactual**: "*the nearest feasible input that flips the decision*" (Exercise 4 below).

## 8. The model card, generated from the code

> **From the deck, *Datasheets for Datasets*:** "*Keep the card in the same repository as the training code, generate its quantitative table from the evaluation script, and make it a merge requirement: a hand-copied table is stale within a week.*"

The numbers come from this notebook. The judgement calls cannot be computed, so they are yours.

> **Exercise (TASK 3):** fill in the card's human fields: the intended use, what is out of scope, and the **SDAIA risk tier** you would argue for, with a one-line reason. Use the deck's *Governance Is Also Law: The Saudi Frame* slide for the four tiers (little or no, limited, high, unacceptable). There is no single right tier. There is only a tier you can defend in a review.

In [ ]:
# TASK 3: replace each "TODO" with your own judgement, in plain words.
INTENDED_USE = "TODO"     # who uses it, for what decision, with what human in the loop
OUT_OF_SCOPE = "TODO"     # uses you would refuse, stated explicitly
RISK_TIER = "TODO"        # little or no / limited / high / unacceptable
RISK_REASON = "TODO"      # one line: why that tier

In [ ]:
OWNER, REVIEWER = "Credit Risk ML team (invented)", "Model Risk Committee (invented)"
REAUDIT = date.today() + timedelta(days=182)

fair = MetricFrame(metrics=METRICS, y_true=y_test, y_pred=y_fair, sensitive_features=A_test).by_group
table = "\n".join(
    f"| {g} | {int(r['n']):,} | {r['selection rate']:.3f} | {r['TPR']:.3f} | {r['FPR']:.3f} | {r['PPV']:.3f} |"
    for g, r in fair.iterrows())
worst_line = ("not computed (TASK 1 skipped)" if worst is None else
              f"{worst.name}: TPR {worst['TPR']:.3f}, 95% interval {worst['95% low']:.2f}-{worst['95% high']:.2f}, "
              f"from {int(worst['positives'])} positive cases")

card = f"""# Model card: credit pre-screening model (teaching example)

**Model details.** Gradient-boosted trees (scikit-learn HistGradientBoosting) with group-specific thresholds
(fairlearn ThresholdOptimizer, `{CHOSEN}`). Owner: {OWNER}. Trained {date.today():%Y-%m-%d}.

**Intended use.** {INTENDED_USE}

**Out of scope.** {OUT_OF_SCOPE}

**Training data.** UCI Adult (1994 US Census), {len(i_train):,} people. The label, income above $50K, is a proxy for
creditworthiness and carries 1994's labour market (historical bias).

**Metrics.** Test accuracy {accuracy_score(y_test, y_fair):.3f} after the threshold fix
({accuracy_score(y_test, y_pred):.3f} before); AUC of the underlying score {roc_auc_score(y_test, p_test):.3f}.
Interpretable baseline (logistic regression) AUC {roc_auc_score(y_test, glass_box.predict_proba(X_test)[:, 1]):.3f}.

**Quantitative analysis (test set, after the threshold fix).**

| group | n | selection rate | TPR | FPR | PPV |
|---|---|---|---|---|---|
{table}

Worst slice with at least 25 positive cases: {worst_line}.

**Fairness criterion adopted.** Equal opportunity (equal TPR across sex). Given up: demographic parity, because the
base rates in the label differ (about 30% vs 11%); and equal FPR. The thresholds differ by sex, which means two
applicants with the same score can be treated differently: this was accepted by {REVIEWER}.

**Ethical considerations.** Removing `sex` does not remove the disparity: the other features predict it with high AUC.
`sex` is kept for auditing only, under access control.

**Risk tier (SDAIA AI Ethics Principles).** {RISK_TIER}. {RISK_REASON}

**Review.** Reviewed by {REVIEWER}. Re-audit due {REAUDIT:%Y-%m-%d}, or earlier after any change to the applicant
population or the label. Contested decisions are logged and reviewed by a person.

*Format after Mitchell et al., "Model Cards for Model Reporting", FAT\\* 2019. Every number above was computed by this notebook.*
"""
Path("model_card.md").write_text(card, encoding="utf-8")
display(Markdown(card))

> **One more governance question.** Under the PDPL, an applicant's record is **personal data**: it needs a lawful basis, a stated purpose, and no more data than the purpose requires. Would you still keep `sex` in the dataset? Section 5 is the argument for keeping it, for auditing only and under access control. Write that decision down too.

## 9. What you did

The deck ends with *A Checklist You Can Use on Monday*. This lab did most of it:

| # | The checklist says | In this lab |
|---|---|---|
| 1 | Write one sentence: who is affected, and how could they be harmed? | The scenario: applicants wrongly declined for credit. |
| 2 | Ask what your label actually measures | Section 1: income in 1994 is a proxy for repayment today. |
| 3 | Check the composition of your data | Section 1: one woman per two men; race groups of a few hundred. |
| 4 | Fit an interpretable baseline and record its score | Section 2: about two AUC points behind, recorded in the card. |
| 5 | Produce the disaggregated table with n and intervals | Section 3, with bootstrap intervals. |
| 6 | Name the fairness criterion and what you traded away | Section 6, written into the card. |
| 7 | Run SHAP or permutation importance and look for a shortcut feature | Section 7: `sex` and `relationship` drive scores directly. |
| 8 | Write the model card, including out-of-scope uses and the worst slice | Section 8. |
| 9 | Get a review from someone who did not build it | Your neighbour: swap cards and review each other's tier. |
| 10 | Set the re-audit date and the monitoring alert on disparity | The date is in the card; the alert is the MLOps lab's drift monitor, pointed at a disparity metric. |

### Exercises

1. **Audit by age.** Cut `age` into bands (`pd.cut(df["age"], [16, 25, 45, 65, 100])`) and rerun section 3 by age band. Which band is worst, and is its interval narrow enough to act on?
2. **Mitigate during training instead.** fairlearn's `ExponentiatedGradient` with a `LogisticRegression` and an `EqualizedOdds()` constraint reduces the disparity while fitting. Compare its accuracy and TPR gap with the threshold fix. Which would you rather defend?
3. **Move the threshold for everyone.** A more permissive pre-screen uses 0.3 instead of 0.5. Recompute section 3: which gaps grow and which shrink?
4. **A counterfactual.** For the applicant in section 7, find the smallest increase in `hours-per-week`, or in `education-num`, that flips the decision. Is that a change the applicant could actually make, and what does that say about recourse?